# Tutorial 07 — Theoretical BER and Monte Carlo Estimation

Closed-form BER expressions exist for AWGN baselines and as *upper bounds* for coded systems. For everything else we estimate BER by counting bit errors in Monte Carlo simulation. This tutorial covers (a) the AWGN baseline, (b) the coded union bound, and (c) building a confidence interval around a Monte-Carlo BER estimate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc
from scipy.stats import binom
rng = np.random.default_rng(0)

## (a) Uncoded BPSK AWGN — closed form
$$\mathrm{BER}_\text{BPSK} = \tfrac{1}{2}\,\mathrm{erfc}\!\left(\sqrt{E_b/N_0}\right).$$

In [ ]:
ebno_db = np.linspace(-2, 10, 25)
g = 10 ** (ebno_db / 10)
ber_bpsk = 0.5 * erfc(np.sqrt(g))
plt.semilogy(ebno_db, ber_bpsk, 'k-', label='BPSK AWGN')
plt.xlabel('Eb/N0 (dB)'); plt.ylabel('BER'); plt.grid(True, which='both', alpha=0.3); plt.legend()

## (b) Convolutional code union bound
$$\mathrm{BER}\;\le\; \tfrac{1}{2}\,\mathrm{erfc}\!\left(\sqrt{d_\mathrm{free}\cdot R\cdot E_b/N_0}\right).$$

In [ ]:
from nsm.modem.ask2 import ber_coded_union_bound, ber_coded_union_full, SPECTRUM_K3_57
bound1 = ber_coded_union_bound(ebno_db, d_free=5, rate=0.5)
bound_full = ber_coded_union_full(ebno_db, SPECTRUM_K3_57, rate=0.5)
plt.semilogy(ebno_db, ber_bpsk, 'k-', label='uncoded BPSK')
plt.semilogy(ebno_db, np.clip(bound1, 1e-12, 1), 'b--', label='K=3 (5,7) dominant term')
plt.semilogy(ebno_db, np.clip(bound_full, 1e-12, 1), 'r:', label='K=3 (5,7) full spectrum')
plt.ylim(1e-8, 1); plt.xlabel('Eb/N0 (dB)'); plt.ylabel('BER')
plt.grid(True, which='both', alpha=0.3); plt.legend()

## (c) BER estimation from a finite Monte Carlo run

Given `e` errors out of `n` Bernoulli trials, the Wilson 95 % CI for the BER is `(p̂ ± z·√(p̂(1−p̂)/n) + z²/(2n)) / (1 + z²/n)` with `z = 1.96`. As `n` grows the relative width shrinks like `1/√n`, but reaching `BER ≈ 10⁻⁵` reliably needs `n ≳ 10⁶` bits.

A simple rule: keep simulating until you accumulate ≥ 100 bit errors at every SNR point.

In [ ]:
def wilson(errors, n, z=1.96):
    p = errors / n; denom = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    half   = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denom
    return centre - half, centre + half
for true_ber in (1e-2, 1e-3, 1e-4):
    n = 100_000
    e = int(rng.binomial(n, true_ber))
    lo, hi = wilson(e, n)
    print(f'true BER {true_ber:.0e} | n={n} | observed {e/n:.2e} | 95% CI ({lo:.2e}, {hi:.2e})')